# 📱 Pocket Studio — on Google Colab

The whole TRIPPEDD animation toolkit, running on Google's computers, controlled from your phone.
7 tools: **Remove BG · Sticker Maker · Trace (→SVG) · Upscale 4x · Loop It · Cartoonize · Auto-Cut to Music**

### 📲 The 3 steps (do this on your phone)
1. **Open this notebook** in Chrome on your phone and tap **Runtime → Run all** (or the ▶ play button, then run each cell top to bottom).
2. **Wait** for the setup cells to finish (~3–5 min the first time — it downloads the tools + AI models). The last cell prints a **`https://....gradio.live`** link.
3. **Tap the gradio.live link** → upload a photo/video → pick a tool → download the result. That's it.

> 💡 Tip: for faster Upscale/Cartoonize, use **Runtime → Change runtime type → T4 GPU** before running. Everything also works on the free CPU runtime, just slower.
> ⚠️ Colab sleeps after ~90 min idle — if the link dies, just re-run the last cell to get a fresh one.


In [ ]:
# ---- Step 1: install the Python packages (Colab already has torch, numpy, PIL, ffmpeg)
!pip install -q "gradio>=4.44" rembg "onnxruntime>=1.17" "librosa==0.10.2" vtracer
print("packages installed")


In [ ]:
# ---- Step 2: pull the animation-kit from GitHub + download the small AI models
import os, sys, glob

KIT_REPO = "https://github.com/mhvnsnt/TRIPPEDD-Production-studios-.git"
if not os.path.isdir("/content/studio"):
    print("cloning animation-kit (21MB, scripts only)...")
    get_ipython().system("git clone --depth 1 --filter=blob:none --sparse " + KIT_REPO + " /content/studio")
    get_ipython().system("git -C /content/studio sparse-checkout set production/tools/animation-kit")
else:
    print("kit already cloned")

KIT = "/content/studio/production/tools/animation-kit"
assert os.path.isdir(KIT + "/rembg"), "kit clone failed — check the repo URL"

# small model weights (rembg models auto-download on first use, ~170MB each)
os.makedirs(KIT + "/upscale/weights", exist_ok=True)
os.makedirs(KIT + "/animeganv3/weights", exist_ok=True)
get_ipython().system("curl -sL -o " + KIT + "/upscale/weights/RealESRGAN_x4plus_anime_6B.pth https://huggingface.co/amd/realesrgan-x4plus-anime-6b/resolve/main/RealESRGAN_x4plus_anime_6B.pth")
get_ipython().system("curl -sL -o " + KIT + "/animeganv3/weights/face_paint_512_v2.pt https://github.com/bryandlee/animegan2-pytorch/raw/main/weights/face_paint_512_v2.pt")
print("weights:", os.path.getsize(KIT + "/upscale/weights/RealESRGAN_x4plus_anime_6B.pth") // 1024, "KB /",
      os.path.getsize(KIT + "/animeganv3/weights/face_paint_512_v2.pt") // 1024, "KB")

# patch hardcoded venv paths in kit scripts -> this notebook's interpreter
patched = 0
for f in glob.glob(KIT + "/**/*.py", recursive=True):
    s = open(f).read()
    s2 = s.replace("/home/hatch/workspace/video-fix-tools/venv/bin/python", sys.executable)
    if s2 != s:
        open(f, "w").write(s2)
        patched += 1
print(f"patched {patched} scripts to use this Python")

# ffmpeg check (Colab ships it)
get_ipython().system("ffmpeg -version 2>/dev/null | head -1")
print("\nsetup complete — KIT =", KIT)


In [ ]:
# ---- Step 3: write the Triller-style auto-cut script (lives in the kit locally; embedded here for Colab)
AUTOCUT_SRC = "#!/usr/bin/env python3\n\"\"\"Triller-style auto-cut: clips cut on music beats, cycled, one edited video out.\n\nUsage:\n  run_autocut.py --clips a.mp4 b.mp4 --music song.mp3 --output out.mp4\n                 [--beats-per-cut 2] [--format vertical] [--max-dur 90]\n\nPipeline: librosa beat_track on the music -> cut points every N beats ->\neach segment pulls the next clip (cycled) from a random offset -> scale/crop\nto output format -> concat -> mux with music -> faststart mp4.\n\"\"\"\nimport argparse, os, random, subprocess, sys, tempfile\n\nVENVPY = sys.executable  # Colab: use this notebook's interpreter\nFF = \"ffmpeg\"\n\n\ndef beat_times(music, max_dur=180):\n    code = (\n        \"import librosa, numpy as np, sys; \"\n        f\"y, sr = librosa.load({music!r}, sr=22050, duration={max_dur}); \"\n        \"tempo, beats = librosa.beat.beat_track(y=y, sr=sr); \"\n        \"ts = librosa.frames_to_time(beats, sr=sr); \"\n        \"print(' '.join(f'{float(t):.3f}' for t in ts))\"\n    )\n    out = subprocess.run([VENVPY, \"-c\", code], capture_output=True, text=True,\n                         timeout=300)\n    if out.returncode != 0:\n        raise RuntimeError(\"beat tracking failed: \" + out.stderr[-500:])\n    return [float(t) for t in out.stdout.split()]\n\n\ndef probe_dur(path):\n    out = subprocess.run(\n        [\"ffprobe\", \"-v\", \"error\", \"-show_entries\", \"format=duration\",\n         \"-of\", \"csv=p=0\", path], capture_output=True, text=True)\n    return float(out.stdout.strip())\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--clips\", nargs=\"+\", required=True)\n    ap.add_argument(\"--music\", required=True)\n    ap.add_argument(\"--output\", required=True)\n    ap.add_argument(\"--beats-per-cut\", type=int, default=2, choices=[1, 2, 4])\n    ap.add_argument(\"--format\", default=\"vertical\", choices=[\"vertical\", \"landscape\"])\n    ap.add_argument(\"--max-dur\", type=float, default=90,\n                    help=\"cap output length in seconds\")\n    ap.add_argument(\"--seed\", type=int, default=None)\n    a = ap.parse_args()\n    if a.seed is not None:\n        random.seed(a.seed)\n\n    beats = beat_times(a.music)\n    if len(beats) < 4:\n        raise RuntimeError(\"too few beats detected in music\")\n    n = a.beats_per_cut\n    cuts = [beats[i] for i in range(0, len(beats), n)]\n    # build segments [cuts[i], cuts[i+1]); total capped\n    segs = []\n    total = 0.0\n    for i in range(len(cuts) - 1):\n        dur = cuts[i + 1] - cuts[i]\n        if total + dur > a.max_dur:\n            dur = a.max_dur - total\n            segs.append((cuts[i], dur))\n            total = a.max_dur\n            break\n        segs.append((cuts[i], dur))\n        total += dur\n    if not segs:\n        raise RuntimeError(\"no segments built\")\n    print(f\"{len(beats)} beats -> {len(segs)} cuts, total {total:.1f}s\", flush=True)\n\n    W, H = (1080, 1920) if a.format == \"vertical\" else (1280, 720)\n    tmp = tempfile.mkdtemp(prefix=\"autocut-\")\n    inputs, filters, vlabels = [], [], []\n    for si, (start, dur) in enumerate(segs):\n        clip = a.clips[si % len(a.clips)]\n        cdur = probe_dur(clip)\n        # random offset inside clip; loop if clip shorter than needed\n        if cdur > dur + 0.5:\n            off = random.uniform(0, cdur - dur - 0.1)\n            loop = []\n        else:\n            off = 0.0\n            loop = [\"-stream_loop\", \"3\"]\n        inputs += loop + [\"-ss\", f\"{off:.3f}\", \"-t\", f\"{dur + 0.2:.3f}\",\n                          \"-i\", clip]\n        vlabels.append(f\"[{si}:v]\")\n        filters.append(\n            f\"[{si}:v]scale={W}:{H}:force_original_aspect_ratio=increase,\"\n            f\"crop={W}:{H},setsar=1,fps=30,\"\n            f\"trim=duration={dur:.3f},setpts=PTS-STARTPTS[v{si}]\")\n    filters.append(\"\".join(f\"[v{i}]\" for i in range(len(segs))) +\n                   f\"concat=n={len(segs)}:v=1:a=0[vout]\")\n    fc = \";\".join(filters)\n    seg_list = os.path.join(tmp, \"segs.mp4\")\n    cmd = [FF, \"-v\", \"error\", \"-y\"] + inputs + [\n        \"-filter_complex\", fc, \"-map\", \"[vout]\",\n        \"-c:v\", \"libx264\", \"-pix_fmt\", \"yuv420p\", \"-preset\", \"fast\",\n        \"-crf\", \"20\", seg_list]\n    r = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)\n    if r.returncode != 0:\n        raise RuntimeError(\"segment build failed: \" + r.stderr[-800:])\n\n    # mux with music, trimmed to video length\n    cmd = [FF, \"-v\", \"error\", \"-y\", \"-i\", seg_list, \"-i\", a.music,\n           \"-map\", \"0:v\", \"-map\", \"1:a\", \"-c:v\", \"copy\", \"-c:a\", \"aac\",\n           \"-b:a\", \"160k\", \"-shortest\", \"-movflags\", \"+faststart\", a.output]\n    r = subprocess.run(cmd, capture_output=True, text=True, timeout=600)\n    if r.returncode != 0:\n        raise RuntimeError(\"mux failed: \" + r.stderr[-800:])\n    print(\"wrote\", a.output, f\"{os.path.getsize(a.output)/1e6:.1f}MB\")\n\n\nif __name__ == \"__main__\":\n    main()\n"
open("/content/run_autocut_colab.py", "w").write(AUTOCUT_SRC)
print("autocut script ready")


In [ ]:
# ---- Step 4: launch Pocket Studio — tap the gradio.live link on your phone ----import gradio as gr, subprocess, os, sys, tempfileKIT = "/content/studio/production/tools/animation-kit"PY = sys.executableAUTOCUT = "/content/run_autocut_colab.py"def run(cmd, timeout=1200):    r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)    if r.returncode != 0:        raise RuntimeError("tool failed: " + (r.stdout + r.stderr)[-600:])    return rdef tmp(suffix):    return tempfile.mktemp(suffix=suffix, dir="/tmp")# 1 — remove backgrounddef bg_remove(img, model, alpha_matting):    out = tmp(".png")    cmd = [PY, f"{KIT}/rembg/run_rembg.py", "--input", img, "--output", out, "--model", model]    if alpha_matting:        cmd.append("--alpha-matting")    run(cmd, timeout=600)    return out# 2 — die-cut stickerdef sticker(img, bg_first, border, shadow):    out = tmp(".png")    cmd = [PY, f"{KIT}/sticker-kit/run_sticker.py", "cutout",           "--input", img, "--output", out, "--border", str(int(border))]    if bg_first:        mid = tmp(".png")        run([PY, f"{KIT}/rembg/run_rembg.py", "--input", img, "--output", mid, "--model", "u2net"], timeout=600)        cmd[cmd.index("--input") + 1] = mid    if shadow:        cmd.append("--shadow")    run(cmd, timeout=600)    return out# 3 — trace to SVGdef trace(img, preset):    out = tmp(".svg")    run([PY, f"{KIT}/tracing/trace_kit.py", "--input", img, "--output", out, "--preset", preset], timeout=600)    return out# 4 — upscale 4xdef upscale(img):    out = tmp(".png")    run([PY, f"{KIT}/upscale/run_upscale.py", "--input", img, "--output", out], timeout=1500)    return out# 5 — hold loopdef loop(img, style, frames):    out = tmp(".mp4")    cmd = [PY, f"{KIT}/loop-kit/run_loop.py", style, "--input", img, "--output", out, "--fps", "30"]    if style == "bob":        cmd += ["--frames", str(int(frames))]    run(cmd, timeout=600)    return out# 6 — cartoonizedef cartoon(img):    out = tmp(".png")    run([PY, f"{KIT}/animeganv3/run_animegan2.py", "--input", img, "--output", out], timeout=1500)    return out# 7 — Triller-style auto-cut to musicdef autocut(clips, music, beats_per_cut, fmt, max_dur):    if not clips or not music:        raise RuntimeError("upload at least 1 clip AND 1 music file")    out = tmp(".mp4")    cmd = [PY, AUTOCUT, "--clips"] + list(clips) + [        "--music", music, "--output", out,        "--beats-per-cut", str(int(beats_per_cut)),        "--format", fmt, "--max-dur", str(float(max_dur))]    run(cmd, timeout=1800)    return outwith gr.Blocks(title="Pocket Studio") as demo:    gr.Markdown("# 📱 Pocket Studio\nUpload → pick a tool → download. Made for phones.")    with gr.Tab("✂️ Remove BG"):        i1 = gr.Image(type="filepath", label="Photo")        m1 = gr.Dropdown(["isnet-general-use", "u2net", "isnet-anime"], value="isnet-general-use", label="Model (isnet = best edges, u2net = fastest)")        a1 = gr.Checkbox(label="Extra edge detail (slower)", value=False)        b1 = gr.Button("Remove background", variant="primary")        o1 = gr.Image(label="Result — long-press to save")        b1.click(bg_remove, [i1, m1, a1], o1)    with gr.Tab("⭐ Sticker"):        i2 = gr.Image(type="filepath", label="Photo")        bf2 = gr.Checkbox(label="Remove background first", value=True)        bo2 = gr.Slider(2, 30, value=7, step=1, label="White border thickness")        sh2 = gr.Checkbox(label="Drop shadow", value=True)        b2 = gr.Button("Make sticker", variant="primary")        o2 = gr.Image(label="Result — long-press to save")        b2.click(sticker, [i2, bf2, bo2, sh2], o2)    with gr.Tab("✏️ Trace → SVG"):        i3 = gr.Image(type="filepath", label="Image")        p3 = gr.Dropdown(["mascot", "logo", "scene"], value="mascot", label="Preset")        b3 = gr.Button("Trace to vector", variant="primary")        o3 = gr.File(label="Download SVG")        b3.click(trace, [i3, p3], o3)    with gr.Tab("🔍 Upscale 4x"):        i4 = gr.Image(type="filepath", label="Image (works best on cartoon/keyframes)")        b4 = gr.Button("Upscale", variant="primary")        o4 = gr.Image(label="Result — long-press to save")        b4.click(upscale, i4, o4)    with gr.Tab("🔁 Loop it"):        i5 = gr.Image(type="filepath", label="Hold frame")        s5 = gr.Radio(["bob", "pingpong"], value="bob", label="Loop style")        f5 = gr.Slider(24, 96, value=48, step=12, label="Frames (bob only)")        b5 = gr.Button("Make loop", variant="primary")        o5 = gr.Video(label="Result")        b5.click(loop, [i5, s5, f5], o5)    with gr.Tab("🎨 Cartoonize"):        i6 = gr.Image(type="filepath", label="Photo")        b6 = gr.Button("Cartoonize", variant="primary")        o6 = gr.Image(label="Result — long-press to save")        b6.click(cartoon, i6, o6)    with gr.Tab("🎬 Auto-cut"):        c7 = gr.File(file_count="multiple", file_types=["video"], label="Clips (2+ recommended)")        mu7 = gr.File(file_types=["audio"], label="Music track")        bp7 = gr.Radio([1, 2, 4], value=2, label="Beats per cut (1 = fast cuts)")        fm7 = gr.Radio(["vertical", "landscape"], value="vertical", label="Format")        md7 = gr.Slider(15, 90, value=60, step=5, label="Max length (sec)")        b7 = gr.Button("Auto-cut to the beat", variant="primary")        o7 = gr.Video(label="Result")        b7.click(autocut, [c7, mu7, bp7, fm7, md7], o7)demo.queue(max_size=4)# NOTE: gradio prints "Running on public URL: https://....gradio.live" when share=True —# tap THAT link on your phone. (launch()'s return shape varies across gradio versions,# so we don't rely on it.)demo.launch(share=True, show_error=True)print("⬆️  TAP THE gradio.live URL ABOVE ON YOUR PHONE")

### 🛠 Tips & troubleshooting
- **Link died?** Colab sleeps when idle — just re-run the last cell for a fresh `gradio.live` link. Your uploads don't persist between sessions.
- **Slow?** Switch to **Runtime → Change runtime type → T4 GPU** and re-run all — Upscale and Cartoonize get much faster.
- **First run downloads ~170MB** of AI models (rembg) — that's normal, one time per session.
- **Auto-cut needs** at least 1 video clip + 1 music file. More clips = more variety.
- Everything here is **free and open source** — the same toolkit in the TRIPPEDD repos (`production/tools/animation-kit/`).
